In [9]:
tabla="cbran10"
dim="dim_resatennom"

In [10]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection2)

connection2.close()




In [11]:
base2

,resatenomedcod,resatenomeddes,resatenomeddescor,resatenomedestregcod
0,01,ALTA,A,1
1,03,RECITA,R,1
2,04,REFERENCIA,R,1


In [12]:
base2.columns

Index(['resatenomedcod', 'resatenomeddes', 'resatenomeddescor',
       'resatenomedestregcod'],
      dtype='object')

In [13]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#connection3.execute('CREATE TEMPORARY TABLE tmp_cmdia10 ()')
base2.to_sql(name=f'{tabla}', con=engine3, if_exists='replace', index=False)


3

In [14]:
#CONEXIONES DESTINO
DB_USER=config("USER2_BDI_DW")
DB_PASSWORD=config("PASS2_BDI_DW")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_DW")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [15]:
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)


3

In [16]:
query=f"""

UPDATE {dim}
SET des_resatennom = t.resatenomeddes,
descor_resatennom =t.resatenomeddescor,
estreg_resatennom =t.resatenomedestregcod

FROM tmp_{tabla} t 
WHERE {dim}.cod_resatennom = t.resatenomedcod;

INSERT INTO {dim} (cod_resatennom, des_resatennom, descor_resatennom, estreg_resatennom) 
SELECT resatenomedcod, resatenomeddes, resatenomeddescor,resatenomedestregcod
FROM tmp_{tabla}
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} 
    WHERE {dim}.cod_resatennom = tmp_{tabla}.resatenomedcod
);

DROP TABLE tmp_{tabla}
"""
c1= text(query)
connection4.execute(c1)
connection4.close()